## Setting environment up for colab

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
from google.colab import userdata

config_path = userdata.get('CONFIG_PATH')

sys.path.append(config_path)

from config import BASE_PATH, DATA_PATH, SRC_PATH, SAVED_CHECKPOINTS_PATH

base_path = BASE_PATH

# Week 2 — Session 1: Advanced Linguistic & Metadata Features

Last week, we focused on:
- **Exploring** our datasets
- **Assessing data quality and sufficiency**
- **Cleaning and saving** consistent versions for modeling

Now, we'll take the next step: **turning raw text and metadata into meaningful features** that a machine learning model can actually learn from.

---

## What We'll Do This Session

In this session, we’ll:
1. Engineer **advanced linguistic features** from the cleaned article text  
   - e.g., word counts, sentence counts, average word length, lexical diversity, readability scores, POS tag ratios.

1. Extract **metadata/source features**  
   - e.g., domain frequency, source credibility signals, publishing date patterns, article length buckets.

1. Combine these features into a structured DataFrame  
   - Ready to use for baseline models next session.

---

## Why This Matters

Effective misinformation detection requires looking beyond what is written to who wrote it and how it is presented. While raw text contains the content, the strongest signals for veracity often lie in the metadata (provenance) and linguistic structure (style). By engineering features that capture source credibility and stylistic signatures—such as emotional intensity or readability—we provide the model with high-level domain expertise that simple text vectorization often misses. This approach allows us to flag potential misinformation based on patterns of deception rather than just keywords.

---

## Libraries We’ll Use

In this session, we'll use a few key Python libraries to engineer and explore our features:

- **`pandas`** → For loading, transforming, and storing our data.
- **`numpy`** → For numerical operations and basic statistics.
- **`nltk`** → For tokenization, word/sentence counts, and linguistic processing.
- **`textstat`** *(optional)* → For calculating readability scores.
- **`matplotlib` & `seaborn`** → For visualizing our feature distributions.

These standard libraries cover everything we need for robust feature engineering pipelines.

In [3]:
!pip install pandas numpy nltk textstat matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 69.8 MB/s eta 0:00:00


In [4]:
!pip install {SRC_PATH}

Processing ./drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/src
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for misinformation_detection: filename=misinformation_detection-1.0.0-py3-none-any.whl size=16007 sha256=1b0589e47977ba8c99af2b5ee628c46e48731312d0e367ae83c1023e0263655e
  Stored in directory: /tmp/pip-ephem-wheel-cache-9y3y1xoh/wheels/46/67/19/6967c24d7ffe355b16d9bd45b2a6576a3fa548c7b58c981aee
Successfully built misinformation_detection


In [8]:
import pandas as pd
import numpy as np
import nltk
import textstat
import matplotlib.pyplot as plt
import seaborn as sns
import misinformation_detection

# Make sure required NLTK resources are available
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

print("Libraries imported and ready!")

Libraries imported and ready!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Data Loading

We already defined the data loading logic in our src folder in the previous week, so we can now just use our predefined function

In [9]:
from misinformation_detection.data import load_fakenewsnet_from_dataframe
from pathlib import Path

fnn_df = load_fakenewsnet_from_dataframe(DATA_PATH / Path("processed/fnn_cleaned.csv"), verbose=True)
fnn_df.sample(3)

Loaded FakeNewsNet cleaned dataset with 13,892 rows
                     id                                              title  \
0  gossipcop-2493749932  Did Miley Cyrus and Liam Hemsworth secretly ge...   
1   gossipcop-941805037  Celebrities Join Tax March in Protest of Donal...   

  label dataset_source    source_domain  \
0  fake      gossipcop  dailymail.co.uk   
1  fake      gossipcop      variety.com   

                                     article_cleaned date_cleaned  \
0  congratulations might be in order for miley cy...   22/06/2018   
1  thousands are taking the streets to protest pr...   15/04/2017   

   source_domain.1 source_domain_grouped  source_Other  ...  source_ew.com  \
0  dailymail.co.uk       dailymail.co.uk         False  ...          False   
1      variety.com           variety.com         False  ...          False   

   source_harpersbazaar.com  source_hollywoodreporter.com  \
0                     False                         False   
1                 

,id,title,label,dataset_source,source_domain,article_cleaned,date_cleaned,source_domain.1,source_domain_grouped,source_Other,...,source_ew.com,source_harpersbazaar.com,source_hollywoodreporter.com,source_inquisitr.com,source_people.com,source_radaronline.com,source_thewrap.com,source_today.com,source_usmagazine.com,source_variety.com
12695,gossipcop-857804,'Bachelorette' Contestant Michael Nance Dies A...,real,gossipcop,bustle.com,here is some sad news for bachelor nation. acc...,30/05/2017,bustle.com,Other,True,...,False,False,False,False,False,False,False,False,False,False
2737,gossipcop-3178329969,"Kristen Stewart and Robert Pattinson, 5 Years ...",fake,gossipcop,eonline.com,it's been five years (and one day) since the s...,25/07/2017,eonline.com,Other,True,...,False,False,False,False,False,False,False,False,False,False
2039,gossipcop-7433410608,The hopeful message hidden in Drake’s “One Dan...,fake,gossipcop,slate.com,"hey slate music clubbers, glad to be back in c...",27/12/2016,slate.com,Other,True,...,False,False,False,False,False,False,False,False,False,False


## Step 1: Basic Linguistic Features

We’ll create simple, interpretable features that describe the writing style of each article:
- Word Count
- Sentence Count
- Average Word Length
- Lexical Richness (Unique Words / Total Words)

These features are good signals for the complexity and repetitiveness of the text.

In [10]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from tqdm import tqdm

# to allow the 'progress apply' function
tqdm.pandas()

# Define helper functions
def sentence_count(text):
    sentences = sent_tokenize(text)
    return len(sentences)

def word_token_features(text):
    tokens = word_tokenize(text)

    word_count = len(tokens)
    if len(tokens) == 0:
        # handle edge cases of 0
        avg_word_length = 0
        lexical_richness = 0
    else:
        unique_tokens = set(tokens)
        avg_word_length = np.mean([len(w) for w in tokens])
        lexical_richness = len(unique_tokens) / len(tokens)

    return pd.Series({
        "word_count": word_count,
        "avg_word_length": avg_word_length,
        "lexical_richness": lexical_richness
      })

# Apply to the cleaned article text
fnn_df["sentence_count"] = fnn_df["article_cleaned"].progress_apply(sentence_count)
fnn_df[[
    "word_count",
    "avg_word_length",
    "lexical_richness"
]] = fnn_df["article_cleaned"].progress_apply(word_token_features)

# Preview
fnn_df[[
    "article_cleaned",
    "word_count",
    "sentence_count",
    "avg_word_length",
    "lexical_richness"
]].head(3)

100%|██████████| 13892/13892 [01:10<00:00, 197.93it/s]


,article_cleaned,word_count,sentence_count,avg_word_length,lexical_richness
0,congratulations might be in order for miley cy...,736.0,37,4.093750,0.432065
1,thousands are taking the streets to protest pr...,352.0,18,4.528409,0.602273
2,we'd venture to say that cindy crawford's daug...,779.0,26,4.245186,0.437741


## Step 2: Advanced Linguistic Features

Now that we've created our **basic text features** (word count, sentence count, average word length, lexical richness),  
we’ll engineer more **advanced linguistic features** that capture deeper aspects of writing style and complexity.

---

### What We’ll Add:
- **Part-of-Speech (POS) Tag Ratios:**  
  Measure the proportion of nouns, verbs, adjectives — useful for detecting stylistic differences between real and fake news.

- **Punctuation & Exclamation Marks:**  
  Clickbait or sensational articles often use more punctuation or excessive exclamation marks.

- **Stopword Ratio:**  
  Shows how much of the text is made up of common stopwords.

- **Readability Scores:**  
  Assess text difficulty using metrics like Flesch Reading Ease.

- **Clickbait Phrase Flag:**  
  Simple binary feature that checks whether an article contains common clickbait phrases  
  (e.g., “you won’t believe”, “what happened next”, “epic fail”).

---

### Why This Matters:
These signals help your model:
- Detect subtle differences in tone, style, or structure.
- Identify repetitive or manipulative language.
- Improve overall robustness of your misinformation detection pipeline.

---

> **Pro Tip:** Don’t overcomplicate it — each feature should have a clear purpose and be easy to interpret!

In [11]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from textstat import flesch_reading_ease
import re
# Make sure resources are ready
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

> NOTE: This cell takes around 10 min to run

In [12]:
# --- Punctuation & Exclamation ---
def punctuation_count(text):
    punctuation = list("!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~")
    return np.sum([text.str.count(re.escape(p)) for p in punctuation], axis=0)

def exclamation_count(text):
    return text.str.count("!")

def exclamation_ratio(df):
    return df["exclamation_count"] / df["word_count"]

# # --- Stopword Ratio ---
def stopword_count(text):
    stop_words = set(stopwords.words('english'))
    return np.sum([text.str.contains(w) for w in stop_words], axis=0)

def stopword_ratio(df):
    return df["stopword_count"] / df["word_count"]

# # --- Readability ---
def flesch_score(text):
    return flesch_reading_ease(text)

# # --- Clickbait Phrase Flag ---
clickbait_phrases = [
    "you won’t believe", "what happened next", "shocking", "this is why",
    "top 10", "goes viral", "are freaking out", "epic fail", "can’t stop laughing"
]

def has_clickbait(text):
    return np.any(
        [text.str.contains(phrase) for phrase in clickbait_phrases],
        axis=0
    )

# # --- POS Tag Ratios ---
def pos_ratios(text):
    tokens = word_tokenize(text)
    if len(tokens) == 0:
        return pd.Series({
            "noun_ratio": 0,
            "verb_ratio": 0,
            "adj_ratio": 0
        })
    tags = nltk.pos_tag(tokens)
    num_nouns = sum(1 for word, tag in tags if tag.startswith('NN'))
    num_verbs = sum(1 for word, tag in tags if tag.startswith('VB'))
    num_adjs  = sum(1 for word, tag in tags if tag.startswith('JJ'))
    total = len(tags)
    return pd.Series({
        "noun_ratio": num_nouns / total,
        "verb_ratio": num_verbs / total,
        "adj_ratio": num_adjs / total
    })

# === Apply to cleaned article text ===
fnn_df = fnn_df.assign(
    # features on the raw text
    punctuation_count = lambda x: punctuation_count(x["article_cleaned"]),
    exclamation_count = lambda x: exclamation_count(x["article_cleaned"]),
    stopword_count = lambda x: stopword_count(x["article_cleaned"]),
    has_clickbait = lambda x: has_clickbait(x["article_cleaned"]),
).assign(
    # features combining columns
    # note: takes dataframe as argument, not series
    exclamation_ratio = lambda x: exclamation_ratio(x),
    stopword_ratio = lambda x: stopword_ratio(x),
).assign(
    # features requiring loops
    flesch_reading_ease = lambda x: x["article_cleaned"].progress_apply(flesch_score),
    **fnn_df["article_cleaned"].progress_apply(pos_ratios)
)

# # Preview
fnn_df[[
    "noun_ratio", "verb_ratio", "adj_ratio",
    "punctuation_count", "exclamation_count", "exclamation_ratio",
    "stopword_ratio", "flesch_reading_ease", "has_clickbait"
]].head(3)

100%|██████████| 13892/13892 [00:33<00:00, 414.30it/s]


,noun_ratio,verb_ratio,adj_ratio,punctuation_count,exclamation_count,exclamation_ratio,stopword_ratio,flesch_reading_ease,has_clickbait
0,0.244565,0.169837,0.082880,104,2,0.002717,0.137228,63.918967,False
1,0.318182,0.153409,0.079545,42,0,0.000000,0.201705,53.148251,False
2,0.278562,0.145058,0.083440,118,2,0.002567,0.124519,57.932812,False


## Step 3. Text preprocessing - Lemmatization

Raw text often contains variations of the same word (e.g., "running", "runs", "ran"). To make our models more efficient, and such that they treat different word forms as the same word, we use **lemmatization** to reduce words to their base form (lemma).

---

### Why Lemmatize?

- Makes your vocabulary **less sparse**.
- Improves **generalization** by grouping word variants.
- Helps the model focus on **meaningful word patterns** instead of superficial variations.


In [13]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Make sure you have WordNet downloaded
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')


lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


> NOTE: This cell takes around 2 min to run

In [14]:
def lemmatize_text(text):
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(lemmatized)

# Apply to cleaned article text
fnn_df["article_lemmatized"] = fnn_df["article_cleaned"].progress_apply(lemmatize_text)

# Quick preview
fnn_df[["article_cleaned", "article_lemmatized"]].head(3)


100%|██████████| 13892/13892 [01:38<00:00, 140.82it/s]


,article_cleaned,article_lemmatized
0,congratulations might be in order for miley cy...,congratulation might be in order for miley cyr...
1,thousands are taking the streets to protest pr...,thousand are taking the street to protest pres...
2,we'd venture to say that cindy crawford's daug...,we 'd venture to say that cindy crawford 's da...


## Step 4: TF-IDF Vectorization

Now that we have clean, lemmatized text, we need to convert it into numbers. We will use **TF-IDF (Term Frequency-Inverse Document Frequency)**.

- **TF**: How often a word appears in a specific article.
- **IDF**: How rare the word is across all articles (punishing words like "the" or "is").

This creates a large matrix of features where every unique word is a column. We will save this vectorizer such that we can reuse it in later sessions.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# Initialize the vectorizer
# max_features=100 limits us to the top 100 words to save memory
tfidf_vectorizer = TfidfVectorizer(max_features=100)

# Fit and transform the lemmatized text
X_tfidf = tfidf_vectorizer.fit_transform(fnn_df["article_lemmatized"])

print(f"TF-IDF Matrix Shape: {X_tfidf.shape}")
# (Rows = Number of articles, Columns = Number of unique words)

# Save the vectorizer for later use in Session 2
vectorizer_path = SAVED_CHECKPOINTS_PATH / Path("tfidf_vectorizer.joblib")
vectorizer_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(tfidf_vectorizer, vectorizer_path)

print(f"TF-IDF Vectorizer saved to {vectorizer_path}")

TF-IDF Matrix Shape: (13892, 100)
TF-IDF Vectorizer saved to /saved_checkpoints/tfidf_vectorizer.joblib


## Step 5: Missing/Valid URL Binary Feature
As we saw last week, not all records contain usable URLs; some are missing, invalid, or marked as unknown.
Rather than ignoring this, we include a binary indicator to show whether each entry has a valid URL or not.

---

## What This Does:
- Assigns 0 when source_domain is either “unknown” or “invalid” (URL missing or not usable).
- Assigns 1 when a valid domain is present (URL usable).
- Adds missing_valid_url column to your DataFrame as a ready-to-use binary feature for modeling.

---

**Why It’s Useful:**

- Can sometimes reveal indirect patterns — e.g., are articles with missing URLs more likely to be fake?
- Use this binary flag can clarify whether other in  the one-hot encoding from the previous feature corresponds to a less common source domain or a missing source domain.

In [16]:
fnn_df["missing_valid_url"] = fnn_df["source_domain"].str.contains("unknown|invalid")

# Preview
fnn_df[[
    "missing_valid_url"
]].head(3)

,missing_valid_url
0,False
1,False
2,False


## Identifying Feature Columns

If you look at the full column list above, you’ll notice:

- The first 7 columns are your **ID, raw text, labels, and metadata**:
  `['id', 'title', 'label', 'source', 'source_domain', 'article_cleaned', 'date_cleaned']`

- **All columns after that** are our **engineered feature columns**:
  - Linguistic features (word count, sentence count, lexical richness, etc.)
  - Readability and style measures
  - Clickbait flag
  - One-hot encoded `source_domain` features
  - Lemmatized text

Names of feature columns will be important.

So, we can **mechanically select** all our final features with:

In [17]:
# Print all column names in the DataFrame
print(f"Total columns: {len(fnn_df.columns)}")
print(fnn_df.columns.tolist()[7:])

Total columns: 41
['source_domain.1', 'source_domain_grouped', 'source_Other', 'source_billboard.com', 'source_dailymail.co.uk', 'source_elle.com', 'source_en.wikipedia.org', 'source_etonline.com', 'source_ew.com', 'source_harpersbazaar.com', 'source_hollywoodreporter.com', 'source_inquisitr.com', 'source_people.com', 'source_radaronline.com', 'source_thewrap.com', 'source_today.com', 'source_usmagazine.com', 'source_variety.com', 'sentence_count', 'word_count', 'avg_word_length', 'lexical_richness', 'punctuation_count', 'exclamation_count', 'stopword_count', 'has_clickbait', 'exclamation_ratio', 'stopword_ratio', 'flesch_reading_ease', 'noun_ratio', 'verb_ratio', 'adj_ratio', 'article_lemmatized', 'missing_valid_url']


## Save All Columns with and without Lemmatization

We will save two versions of the dataset, one with and another without lemmatization:
- Original raw fields `['id', 'title', 'label', 'source', 'source_domain', 'article_cleaned', 'date_cleaned']`
- Target label
- All engineered linguistic features
- One-hot encoded `source_domain` features

While we might not need all the columns during training, it is always better to drop the unnecessary columns during training but keep them in case they are needed for later inspection (e.g. interpretability analysis)

In [18]:
# Save the entire DataFrame with all columns
without_lemmatization = fnn_df.columns.tolist()
without_lemmatization.remove("article_lemmatized")

print(without_lemmatization)

output_path = DATA_PATH / Path("processed/fnn_features_all_columns.csv")
fnn_df[without_lemmatization].to_csv(output_path, index=False)

print(f"Full DataFrame saved to: {output_path}")
print(f"Total rows: {len(fnn_df)}, Total columns: {len(without_lemmatization)}")

['id', 'title', 'label', 'dataset_source', 'source_domain', 'article_cleaned', 'date_cleaned', 'source_domain.1', 'source_domain_grouped', 'source_Other', 'source_billboard.com', 'source_dailymail.co.uk', 'source_elle.com', 'source_en.wikipedia.org', 'source_etonline.com', 'source_ew.com', 'source_harpersbazaar.com', 'source_hollywoodreporter.com', 'source_inquisitr.com', 'source_people.com', 'source_radaronline.com', 'source_thewrap.com', 'source_today.com', 'source_usmagazine.com', 'source_variety.com', 'sentence_count', 'word_count', 'avg_word_length', 'lexical_richness', 'punctuation_count', 'exclamation_count', 'stopword_count', 'has_clickbait', 'exclamation_ratio', 'stopword_ratio', 'flesch_reading_ease', 'noun_ratio', 'verb_ratio', 'adj_ratio', 'missing_valid_url']
Full DataFrame saved to: /content/drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/week_2/sessions/../../data/processed/fnn_features_all_columns.csv
Total rows: 13892, Total columns: 40


In [19]:
# Save the DataFrame with the new lemmatized text column
output_path = DATA_PATH / Path("processed/fnn_lemmatized.csv")
fnn_df.to_csv(output_path, index=False)

print(f"Saved lemmatized dataset to: {output_path}")

Saved lemmatized dataset to: /content/drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/week_2/sessions/../../data/processed/fnn_lemmatized.csv


## Add the Feature Engineering pipeline to our package

> 📝: This function will be placed into src

Now we should organize our code into a singular function holding the entire feature engineering pipeline. We will add this function to `/src/misinformation_detection/data_preprocessing/feature_engineering.py`

In [20]:
import os

# # --- basic features ---
def sentence_count(sentences):
    return len(sentences)

def word_count(tokens):
    return len(tokens)

def avg_word_length(tokens):
    if len(tokens) == 0:
        return 0
    return np.mean([len(w) for w in tokens])

def lexical_richness(tokens):
    if len(tokens) == 0:
        return 0
    return len(set(tokens)) / len(tokens)

def _basic_features_from_row(row):
    sentences = sent_tokenize(row["article_cleaned"])
    tokens = word_tokenize(row["article_cleaned"])
    return pd.Series({
        "word_count": word_count(tokens),
        "avg_word_length": avg_word_length(tokens),
        "lexical_richness": lexical_richness(tokens),
        "sentence_count": sentence_count(tokens),
      })

def basic_lingusitic_features(df):
    return df.assign(**df.progress_apply(_basic_features_from_row, axis=1))

basic_lingusitic_features(fnn_df.sample(2))

100%|██████████| 2/2 [00:00<00:00, 84.78it/s]


,id,title,label,dataset_source,source_domain,article_cleaned,date_cleaned,source_domain.1,source_domain_grouped,source_Other,...,stopword_count,has_clickbait,exclamation_ratio,stopword_ratio,flesch_reading_ease,noun_ratio,verb_ratio,adj_ratio,article_lemmatized,missing_valid_url
6642,gossipcop-911821,"Margot Robbie, Salma Hayek Chill Out at Indepe...",real,gossipcop,wwd.com,"thank you so much for seeing it, salma hayek ...",04/03/2018,wwd.com,Other,True,...,94,False,0.001264,0.118837,61.069476,0.243995,0.193426,0.098609,"thank you so much for seeing it , salma hayek ...",False
5829,gossipcop-935048,Nikki Bella Says She Only Shared a Bed With Jo...,real,gossipcop,etonline.com,nikki bella is giving fans a glimpse into what...,UNKNOWN_DATE,etonline.com,etonline.com,False,...,98,False,0.000000,0.146487,77.454646,0.162930,0.189836,0.074738,nikki bella is giving fan a glimpse into what ...,False


In [21]:
# # --- advanced features ---
# ## --- Punctuation & Exclamation ---
def punctuation_count(text):
    punctuation = list("!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~")
    return np.sum([text.str.count(re.escape(p)) for p in punctuation], axis=0)

def exclamation_count(text):
    return str(text).count("!")

def exclamation_ratio(df):
    return df["exclamation_count"] / df["word_count"]

## --- Stopword Ratio ---
def stopword_count(text):
    stop_words = set(stopwords.words('english'))
    return np.sum([text.str.contains(w) for w in stop_words], axis=0)

def stopword_ratio(df):
    return df["stopword_count"] / df["word_count"]

## --- Readability ---
def flesch_score(text):
    return flesch_reading_ease(text)

## --- Clickbait Phrase Flag ---
clickbait_phrases = [
    "you won’t believe", "what happened next", "shocking", "this is why",
    "top 10", "goes viral", "are freaking out", "epic fail", "can’t stop laughing"
]

def has_clickbait(text):
    return np.any(
        [text.str.contains(phrase) for phrase in clickbait_phrases],
        axis=0
    )

## --- POS Tag Ratios ---
def pos_ratios(text):
    tokens = word_tokenize(text)
    if len(tokens) == 0:
        return pd.Series({
            "noun_ratio": 0,
            "verb_ratio": 0,
            "adj_ratio": 0
        })
    tags = nltk.pos_tag(tokens)
    num_nouns = sum(1 for word, tag in tags if tag.startswith('NN'))
    num_verbs = sum(1 for word, tag in tags if tag.startswith('VB'))
    num_adjs  = sum(1 for word, tag in tags if tag.startswith('JJ'))
    total = len(tags)
    return pd.Series({
        "noun_ratio": num_nouns / total,
        "verb_ratio": num_verbs / total,
        "adj_ratio": num_adjs / total
    })

def advanced_linguistic_features(df):
    nltk.download('punkt')
    nltk.download('averaged_perceptron_tagger_eng')
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))
    return (
      df.assign(
          # features on the raw text
          punctuation_count = lambda x: punctuation_count(x["article_cleaned"]),
          exclamation_count = lambda x: exclamation_count(x["article_cleaned"]),
          stopword_count = lambda x: stopword_count(x["article_cleaned"]),
          has_clickbait = lambda x: has_clickbait(x["article_cleaned"]),
      ).assign(
          # features combining columns
          exclamation_ratio = lambda x: exclamation_ratio(x), # note: takes dataframe as argument, not series
          stopword_ratio = lambda x: stopword_ratio(x),
      ).assign(
          # features requiring loops
          flesch_reading_ease = lambda x: x["article_cleaned"].progress_apply(flesch_score),
          **df["article_cleaned"].progress_apply(pos_ratios)
      )
    )
advanced_linguistic_features(fnn_df.sample(2))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
100%|██████████| 2/2 [00:00<00:00, 203.02it/s]


,id,title,label,dataset_source,source_domain,article_cleaned,date_cleaned,source_domain.1,source_domain_grouped,source_Other,...,stopword_count,has_clickbait,exclamation_ratio,stopword_ratio,flesch_reading_ease,noun_ratio,verb_ratio,adj_ratio,article_lemmatized,missing_valid_url
1610,gossipcop-4322858010,Analyzing Every Song on Taylor Swift's Reputation,fake,gossipcop,time.com,taylor swift s sixth studio album reputation f...,10/11/2017,time.com,Other,True,...,122,True,0.0,0.054295,66.644495,0.221629,0.142857,0.098798,taylor swift s sixth studio album reputation f...,False
6157,gossipcop-878994,Janet Jackson breaks down singing abuse track ...,real,gossipcop,dailymail.co.uk,janet jackson broke down in tears onstage in h...,11/09/2017,dailymail.co.uk,dailymail.co.uk,False,...,96,False,0.0,0.078240,61.401905,0.260799,0.159739,0.066830,janet jackson broke down in tear onstage in ho...,False


In [22]:
# # --- source domain features ---
def missing_valid_url(source_domain):
  return source_domain.str.contains("unknown|invalid")

def source_domain_one_hot_encoding(df):
    # Count top 15 domains
    top_domains = df["source_domain"].value_counts().nlargest(15).index.tolist()

    # Assign 'Other' to all less common domains
    source_domain_grouped = df["source_domain"].apply(
        lambda x: x if x in top_domains else "Other"
    )

    # One-hot encode the grouped domain column
    source_dummies = pd.get_dummies(source_domain_grouped, prefix="source")

    return source_dummies

def source_domain_features(df):
    return df.assign(
        missing_valid_url = lambda x: missing_valid_url(x["source_domain"]),

    )
source_domain_features(fnn_df.sample(2))

,id,title,label,dataset_source,source_domain,article_cleaned,date_cleaned,source_domain.1,source_domain_grouped,source_Other,...,stopword_count,has_clickbait,exclamation_ratio,stopword_ratio,flesch_reading_ease,noun_ratio,verb_ratio,adj_ratio,article_lemmatized,missing_valid_url
1243,gossipcop-8046229859,Rachel Weisz and Daniel Craig have the most in...,fake,gossipcop,harpersbazaar.com,daniel craig and rachel weisz have confirmed t...,20/04/2018,harpersbazaar.com,harpersbazaar.com,False,...,100,False,0.001333,0.133333,73.469080,0.212000,0.164000,0.081333,daniel craig and rachel weisz have confirmed t...,False
13887,politifact11627,Senator Bernie Sanders on Democratic Socialism...,real,politifact,berniesanders.com,thank you for donating! together we are buildi...,UNKNOWN_DATE,berniesanders.com,Other,True,...,44,False,0.019608,0.862745,64.924545,0.176471,0.176471,0.098039,thank you for donating ! together we are build...,False


In [23]:
def lemmatize_text(text):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(lemmatized)

def lemmatize_text_features(df):
    # Make sure you have WordNet downloaded
    nltk.download('punkt')
    nltk.download('wordnet')
    return df.assign(article_lemmatized = lambda x: x["article_cleaned"].apply(lemmatize_text))

lemmatize_text_features(fnn_df.sample(2))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,id,title,label,dataset_source,source_domain,article_cleaned,date_cleaned,source_domain.1,source_domain_grouped,source_Other,...,stopword_count,has_clickbait,exclamation_ratio,stopword_ratio,flesch_reading_ease,noun_ratio,verb_ratio,adj_ratio,article_lemmatized,missing_valid_url
2856,gossipcop-2937874357,"Selena Gomez and Justin Bieber Are ""Talking Ba...",fake,gossipcop,lifeandstylemag.com,there wasn t just romance in the air when just...,07/03/2018,lifeandstylemag.com,Other,True,...,75,False,0.010959,0.205479,69.928199,0.263014,0.145205,0.082192,there wasn t just romance in the air when just...,False
9204,gossipcop-890014,'Stranger Things' star Charlie Heaton denied e...,real,gossipcop,torontosun.com,you can save this article by registering for f...,UNKNOWN_DATE,torontosun.com,Other,True,...,73,False,0.001647,0.120264,54.853625,0.285008,0.144975,0.087315,you can save this article by registering for f...,False


In [24]:
def log_shape(df, text = ""):
    print(f"{text} {df.shape}")
    return df

def engineer_features(df, lemmatize=False, save_path=None):
    """
    Perform comprehensive feature engineering on the input DataFrame containing news articles.

    This function applies multiple linguistic and source-related transformations to enrich the dataset
    with meaningful features for downstream analysis or machine learning modeling.

    Features engineered include:

    1. Basic linguistic features extracted from the article text:
       - Word count
       - Sentence count
       - Average word length
       - Lexical richness (unique word ratio)

    2. Advanced linguistic features including:
       - Part-of-speech (POS) tag ratios: noun, verb, adjective proportions
       - Punctuation and exclamation counts and ratios
       - Stopword usage ratio
       - Readability score (Flesch Reading Ease)
       - Clickbait phrase presence flag

    3. Source domain encoding:
       - One-hot encoding of the top 15 most frequent source domains with others grouped as 'Other'

    4. URL validity flag:
       - Binary indicator flag marking whether a URL is valid or missing/invalid based on extracted domain

    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame with at least the following columns:
         - "article_cleaned": cleaned plain-text article content for linguistic features
         - "source_domain": domain extracted from article URL for source encoding

    Returns:
    --------
    pandas.DataFrame
        The input DataFrame augmented with newly engineered feature columns ready for model use.
    """

    return (
        df
        .pipe(log_shape, "input dataframe")
        .pipe(basic_lingusitic_features)
        .pipe(log_shape, "with basic features")
        .pipe(advanced_linguistic_features)
        .pipe(log_shape, "advanced linguistic features")
        .pipe(source_domain_features)
        .pipe(log_shape, "with source domain features")
    )

    # Call the new function if requested
    if lemmatize:
        df = lemmatize_text_features(df)

    # Save if requested
    if save_path is not None:
        save_path = os.path.abspath(save_path)
        if hasattr(save_path, 'parent'):
            save_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(save_path, index=False)
        print(f"Feature engineered data saved to {save_path}")

    return df

In [25]:
fnn_df = load_fakenewsnet_from_dataframe(DATA_PATH / Path("processed/fnn_cleaned.csv"), verbose=True)
# test function on a sample of 150 rows
(
    fnn_df
    .iloc[:150]
    .pipe(engineer_features)
)

Loaded FakeNewsNet cleaned dataset with 13,892 rows
                     id                                              title  \
0  gossipcop-2493749932  Did Miley Cyrus and Liam Hemsworth secretly ge...   
1   gossipcop-941805037  Celebrities Join Tax March in Protest of Donal...   

  label dataset_source    source_domain  \
0  fake      gossipcop  dailymail.co.uk   
1  fake      gossipcop      variety.com   

                                     article_cleaned date_cleaned  \
0  congratulations might be in order for miley cy...   22/06/2018   
1  thousands are taking the streets to protest pr...   15/04/2017   

   source_domain.1 source_domain_grouped  source_Other  ...  source_ew.com  \
0  dailymail.co.uk       dailymail.co.uk         False  ...          False   
1      variety.com           variety.com         False  ...          False   

   source_harpersbazaar.com  source_hollywoodreporter.com  \
0                     False                         False   
1                 

100%|██████████| 150/150 [00:00<00:00, 220.34it/s]
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


with basic features (150, 29)


100%|██████████| 150/150 [00:00<00:00, 482.58it/s]

advanced linguistic features (150, 39)
with source domain features (150, 40)


,id,title,label,dataset_source,source_domain,article_cleaned,date_cleaned,source_domain.1,source_domain_grouped,source_Other,...,exclamation_count,stopword_count,has_clickbait,exclamation_ratio,stopword_ratio,flesch_reading_ease,noun_ratio,verb_ratio,adj_ratio,missing_valid_url
0,gossipcop-2493749932,Did Miley Cyrus and Liam Hemsworth secretly ge...,fake,gossipcop,dailymail.co.uk,congratulations might be in order for miley cy...,22/06/2018,dailymail.co.uk,dailymail.co.uk,False,...,0,101,False,0.0,0.137228,63.918967,0.244565,0.169837,0.082880,False
1,gossipcop-941805037,Celebrities Join Tax March in Protest of Donal...,fake,gossipcop,variety.com,thousands are taking the streets to protest pr...,15/04/2017,variety.com,variety.com,False,...,0,71,False,0.0,0.201705,53.148251,0.318182,0.153409,0.079545,False
2,gossipcop-2547891536,Cindy Crawford's daughter Kaia Gerber wears a ...,fake,gossipcop,dailymail.co.uk,we'd venture to say that cindy crawford's daug...,18/03/2016,dailymail.co.uk,dailymail.co.uk,False,...,0,97,False,0.0,0.124519,57.932812,0.278562,0.145058,0.083440,False
3,gossipcop-5476631226,Full List of 2018 Oscar Nominations – Variety,fake,gossipcop,variety.com,oscar nominations for the 90th annual awards w...,23/01/2018,variety.com,variety.com,False,...,0,67,False,0.0,0.060252,16.984443,0.433453,0.079137,0.105216,False
4,gossipcop-5189580095,Here's What Really Happened When JFK Jr. Met P...,fake,gossipcop,townandcountrymag.com,"during the summer of 1995, john f. kennedy jr....",29/06/2017,townandcountrymag.com,Other,True,...,0,83,False,0.0,0.190367,69.824205,0.224771,0.201835,0.080275,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,gossipcop-9627008778,Inside Ben Affleck and Jennifer Garner's Divor...,fake,gossipcop,people.com,ben affleck and jennifer garner are moving on ...,UNKNOWN_DATE,people.com,people.com,False,...,0,88,False,0.0,0.128843,67.000541,0.253294,0.171303,0.065886,False
146,gossipcop-576948808,Brad Pitt and uni professor Neri Oxman ‘have b...,fake,gossipcop,thesun.co.uk,building foundations brad pitt and uni profess...,10/04/2018,thesun.co.uk,Other,True,...,0,100,False,0.0,0.125000,47.706667,0.270000,0.167500,0.082500,False
147,gossipcop-1702570445,Is Mariah Carey Forbidding Bryan Tanaka From S...,fake,gossipcop,inquisitr.com,has mariah carey forbidden boyfriend bryan tan...,10/09/2017,inquisitr.com,inquisitr.com,False,...,0,83,False,0.0,0.153137,54.984261,0.250923,0.195572,0.094096,False
148,gossipcop-2965059427,Hoda Kotb Says She Still 'Keeps in Touch' with...,fake,gossipcop,people.com,matt lauer may no longer be a part of the toda...,UNKNOWN_DATE,people.com,people.com,False,...,0,90,False,0.0,0.126939,64.465893,0.208745,0.201693,0.069111,False


## [💎 Additional Credit] Pytest

Check out the tests folder to see how we test this function